# YOLOv11n ESP32-S3 WROOM-1 Optimization Pipeline
## Single-Class Detection for Edge AI Deployment

**Architecture Overview:**
- Model: YOLOv11n (2.6M parameters, 640px input)
- Target Hardware: ESP32-S3 WROOM-1 Dev Kit
- Optimization Strategy: Post-Training Quantization (PTQ) to INT8
- Performance Goal: Competitive mAP[0.5:0.95] with optimized inference on ESP32-S3

**Pipeline Stages:**
1. Environment Configuration & Dataset Preparation
2. Custom Data Augmentation for 640x640 Resolution
3. Training with ESP32-Optimized Hyperparameters
4. Validation & Performance Benchmarking
5. PTQ Export & ESP-IDF Conversion
6. ESP32-S3 Deployment Utilities

**ESP32-S3 WROOM-1 Specifications:**
- Xtensa LX7 Dual-Core @ 240MHz
- 512KB SRAM + 8MB PSRAM (external)

- Vector instructions for ML acceleration- Ultralytics YOLOv11: https://docs.ultralytics.com/models/yolo11/

- Target inference: ~200-500ms @ 640x640 (consider reducing to 416x416 for real-time)- Espressif Detection: https://github.com/espressif/esp-detection

**References:**

In [ ]:
# System and Environment Information
import sys
import os
import platform
from pathlib import Path

print(f"Python Version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Working Directory: {os.getcwd()}")
print(f"GPU Available: {os.system('nvidia-smi > /dev/null 2>&1') == 0}")

# Set project root
PROJECT_ROOT = Path("/home/ubuntu/edge-ai-vineyard-monitoring/YOLO")
os.chdir(PROJECT_ROOT)
print(f"Project Root: {PROJECT_ROOT}")

In [ ]:
# Import Core Libraries
import torch
import ultralytics
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import yaml
import json
from datetime import datetime
import shutil
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

In [ ]:
# Dataset Configuration
DATASET_PATH = PROJECT_ROOT / "Dataset.v1.yolov11"
DATA_YAML = DATASET_PATH / "data.yaml"

# Verify dataset structure
assert DATASET_PATH.exists(), f"Dataset not found at {DATASET_PATH}"
assert DATA_YAML.exists(), f"data.yaml not found at {DATA_YAML}"

# Load and display dataset configuration
with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)

print("Dataset Configuration:")
print(f"├── Classes: {data_config['nc']}")
print(f"├── Class Names: {data_config['names']}")
print(f"├── Train Images: {DATASET_PATH / 'train' / 'images'}")
print(f"├── Validation Images: {DATASET_PATH / 'valid' / 'images'}")
print(f"└── Test Images: {DATASET_PATH / 'test' / 'images'}")

# Count images in each split
train_count = len(list((DATASET_PATH / 'train' / 'images').glob('*.jpg')))
val_count = len(list((DATASET_PATH / 'valid' / 'images').glob('*.jpg')))
test_count = len(list((DATASET_PATH / 'test' / 'images').glob('*.jpg')))

print(f"\nDataset Statistics:")
print(f"├── Training: {train_count} images")
print(f"├── Validation: {val_count} images")
print(f"├── Test: {test_count} images")
print(f"└── Total: {train_count + val_count + test_count} images")

### ESP32-Optimized Augmentation Configuration
Following Espressif's detection guidelines for edge deployment

In [ ]:
# Create ESP32-Optimized Hyperparameter Configuration
# Based on Espressif detection repository guidelines for edge deployment

esp32_hyperparameters = {
    # Model Configuration
    'imgsz': 640,  # Input image size (640x640 for YOLOv11n)
    'batch': 16,   # Batch size (adjust based on GPU memory)
    'epochs': 150,  # Training epochs
    'patience': 30, # Early stopping patience
    
    # Optimization Parameters
    'optimizer': 'AdamW',  # Optimizer (AdamW recommended for YOLOv11)
    'lr0': 0.001,         # Initial learning rate
    'lrf': 0.01,          # Final learning rate (lr0 * lrf)
    'momentum': 0.937,    # SGD momentum/Adam beta1
    'weight_decay': 0.0005,  # Weight decay
    'warmup_epochs': 3.0,    # Warmup epochs
    'warmup_momentum': 0.8,  # Warmup initial momentum
    'warmup_bias_lr': 0.1,   # Warmup initial bias lr
    
    # Augmentation (Conservative for Edge Deployment)
    'hsv_h': 0.015,  # HSV-Hue augmentation (0.0-1.0)
    'hsv_s': 0.5,    # HSV-Saturation augmentation
    'hsv_v': 0.3,    # HSV-Value augmentation
    'degrees': 10.0,  # Rotation augmentation (degrees)
    'translate': 0.1, # Translation augmentation (fraction)
    'scale': 0.3,     # Scaling augmentation (gain)
    'shear': 2.0,     # Shear augmentation (degrees)
    'perspective': 0.0, # Perspective augmentation (0.0 disabled for stability)
    'flipud': 0.0,    # Vertical flip probability
    'fliplr': 0.5,    # Horizontal flip probability
    'mosaic': 1.0,    # Mosaic augmentation probability
    'mixup': 0.1,     # MixUp augmentation probability
    'copy_paste': 0.0, # Copy-paste augmentation probability
    
    # Loss Configuration
    'box': 7.5,      # Box loss gain
    'cls': 0.5,      # Class loss gain (single-class, can be lower)
    'dfl': 1.5,      # DFL loss gain
    
    # Model Pruning for Edge (Conservative settings)
    'dropout': 0.0,   # Dropout rate (0.0 for inference optimization)
    'label_smoothing': 0.0,  # Label smoothing epsilon
    
    # Validation & Saving
    'val': True,      # Validate during training
    'save': True,     # Save checkpoints
    'save_period': -1,  # Save checkpoint every x epochs (-1 to disable)
    'plots': True,    # Generate training plots
    'device': 0 if torch.cuda.is_available() else 'cpu',  # Training device
    'workers': 8,     # DataLoader workers
    'exist_ok': True, # Overwrite existing project
    'pretrained': True,  # Use pretrained weights
    'verbose': True,  # Verbose output
    
    # ESP32-Specific Optimizations
    'half': False,    # Use FP16 training (disable for PTQ compatibility)
    'amp': False,     # Automatic Mixed Precision (disable for stable PTQ)
    'fraction': 1.0,  # Dataset fraction to train on
    'profile': False, # Profile ONNX and TensorRT speed
    'freeze': None,   # Freeze layers (None = train all)
    'multi_scale': False,  # Multi-scale training (disable for fixed 640x640)
    
    # Model Architecture
    'name': 'yolo11n',  # Model variant
    'task': 'detect',   # Task type
}

# Save hyperparameters
hyp_path = PROJECT_ROOT / 'hyp_esp32_yolo11n.yaml'
with open(hyp_path, 'w') as f:
    yaml.dump(esp32_hyperparameters, f, default_flow_style=False, sort_keys=False)

print("ESP32-Optimized Hyperparameters:")
print("=" * 60)
for key, value in esp32_hyperparameters.items():
    print(f"{key:20s}: {value}")
print("=" * 60)
print(f"✓ Configuration saved to: {hyp_path}")

In [ ]:
# Initialize YOLOv11n Model
# 2.6M parameters, optimized for edge deployment

model = YOLO('yolo11n.pt')  # Load pretrained YOLOv11n weights

# Display model architecture summary
print("YOLOv11n Model Architecture:")
print("=" * 60)
print(model.model)
print("=" * 60)

# Model statistics
total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

print(f"\nModel Statistics:")
print(f"├── Total Parameters: {total_params:,} ({total_params/1e6:.2f}M)")
print(f"├── Trainable Parameters: {trainable_params:,}")
print(f"├── Input Size: 640x640x3")
print(f"├── Model Type: YOLOv11n (Nano)")
print(f"└── Task: Single-Class Object Detection")

# Verify model is on correct device
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"\nTraining Device: {device}")

In [ ]:
# Visualize Training Results
from IPython.display import Image, display

# Training results paths
results_dir = PROJECT_ROOT / 'runs' / 'detect' / 'yolo11n_leaf_esp32'

# Display training curves
print("Training Performance Metrics:")
print("=" * 60)

if (results_dir / 'results.png').exists():
    display(Image(filename=str(results_dir / 'results.png')))
    print("✓ Training curves displayed")
else:
    print("⚠ Results plot not found - training may still be in progress")

# Display confusion matrix
if (results_dir / 'confusion_matrix.png').exists():
    print("\nConfusion Matrix:")
    display(Image(filename=str(results_dir / 'confusion_matrix.png')))
    print("✓ Confusion matrix displayed")

# Display training batch examples
if (results_dir / 'train_batch0.jpg').exists():
    print("\nTraining Batch Example:")
    display(Image(filename=str(results_dir / 'train_batch0.jpg')))
    print("✓ Training batch visualization displayed")

In [ ]:
# Load Best Model for Validation
best_model_path = results_dir / 'weights' / 'best.pt'

if best_model_path.exists():
    model_best = YOLO(str(best_model_path))
    print(f"✓ Loaded best model from: {best_model_path}")
else:
    print("⚠ Best model not found - using current model")
    model_best = model

# Comprehensive Validation on Test Set
print("\nRunning Comprehensive Validation...")
print("=" * 60)

val_results = model_best.val(
    data=str(DATA_YAML),
    split='test',  # Validate on test set
    imgsz=640,
    batch=16,
    conf=0.25,     # Confidence threshold
    iou=0.6,       # NMS IoU threshold
    plots=True,
    save_json=True,  # Save results in JSON format for analysis
    project='runs/detect',
    name='val_esp32',
    exist_ok=True,
)

print("=" * 60)
print("\nValidation Metrics:")
print("=" * 60)
print(f"mAP@0.5:0.95: {val_results.box.map:.4f}")
print(f"mAP@0.5:     {val_results.box.map50:.4f}")
print(f"mAP@0.75:    {val_results.box.map75:.4f}")
print(f"Precision:   {val_results.box.mp:.4f}")
print(f"Recall:      {val_results.box.mr:.4f}")
print("=" * 60)

# Save validation metrics
val_metrics = {
    'mAP_50_95': float(val_results.box.map),
    'mAP_50': float(val_results.box.map50),
    'mAP_75': float(val_results.box.map75),
    'precision': float(val_results.box.mp),
    'recall': float(val_results.box.mr),
    'timestamp': datetime.now().isoformat(),
}

metrics_path = PROJECT_ROOT / 'validation_metrics_esp32.json'
with open(metrics_path, 'w') as f:
    json.dump(val_metrics, f, indent=2)

print(f"✓ Validation metrics saved to: {metrics_path}")

In [ ]:
# Visualize Predictions on Test Set
def visualize_predictions(model, image_paths, num_samples=6):
    """Visualize model predictions on test images"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(image_paths[:num_samples]):
        # Run inference
        results = model(str(img_path), imgsz=640, conf=0.25, verbose=False)[0]
        
        # Plot results
        img_with_boxes = results.plot()
        img_rgb = cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB)
        
        axes[idx].imshow(img_rgb)
        axes[idx].set_title(f"{Path(img_path).name}\nBoxes: {len(results.boxes)}", 
                           fontsize=10)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('test_predictions_esp32.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Test predictions saved as 'test_predictions_esp32.png'")

# Visualize predictions
test_samples = list((DATASET_PATH / 'test' / 'images').glob('*.jpg'))[:6]
visualize_predictions(model_best, test_samples)

In [ ]:
# Step 1: Export to ONNX Format
print("Exporting YOLOv11n to ONNX Format...")
print("=" * 60)

onnx_path = model_best.export(
    format='onnx',
    imgsz=640,
    simplify=True,  # Simplify ONNX graph
    dynamic=False,   # Static input size for ESP32
    opset=12,        # ONNX opset version (compatible with TFLite conversion)
)

print(f"✓ ONNX model exported to: {onnx_path}")

# Verify ONNX model
import onnx
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("✓ ONNX model verification passed")

# Display ONNX model info
print(f"\nONNX Model Information:")
print(f"├── Input Shape: {onnx_model.graph.input[0].type.tensor_type.shape}")
print(f"├── IR Version: {onnx_model.ir_version}")
print(f"└── Opset Version: {onnx_model.opset_import[0].version}")
print("=" * 60)

In [ ]:
# Step 3: Convert ONNX to TensorFlow SavedModel
print("Converting ONNX to TensorFlow SavedModel...")
print("=" * 60)

try:
    import onnx
    from onnx_tf.backend import prepare
    
    # Load ONNX model
    onnx_model = onnx.load(onnx_path)
    
    # Convert to TensorFlow
    tf_rep = prepare(onnx_model)
    
    # Export as SavedModel
    savedmodel_path = PROJECT_ROOT / 'yolo11n_savedmodel'
    tf_rep.export_graph(str(savedmodel_path))
    
    print(f"✓ TensorFlow SavedModel exported to: {savedmodel_path}")
    print("=" * 60)
    
except Exception as e:
    print(f"⚠ ONNX to TensorFlow conversion failed: {e}")
    print("Attempting alternative TFLite export method...")
    
    # Alternative: Direct TFLite export from Ultralytics (if supported)
    try:
        tflite_path = model_best.export(
            format='tflite',
            imgsz=640,
            int8=False,  # First export FP32 version
        )
        print(f"✓ TFLite FP32 model exported to: {tflite_path}")
    except Exception as e2:
        print(f"⚠ Direct TFLite export also failed: {e2}")
        print("Manual conversion will be required")
    
    savedmodel_path = None
    print("=" * 60)

### Alternative: Direct TFLite INT8 Export using Ultralytics
If ONNX conversion fails, use Ultralytics built-in TFLite export

## 6. ESP32-S3 WROOM-1 Deployment
### Conversion to ESP-DL Format for ESP32-S3 Hardware

In [ ]:
# Create ESP32 Deployment Package
print("Creating ESP32 Deployment Package...")
print("=" * 60)

# Create deployment directory
deployment_dir = PROJECT_ROOT / 'esp32_deployment_package'
deployment_dir.mkdir(exist_ok=True)

# Copy essential files
import shutil

files_to_copy = {
    'Model Weights': best_model_path,
    'ONNX Model': onnx_path,
    'Calibration Data': calib_path,
    'Hyperparameters': hyp_path,
    'Validation Metrics': metrics_path,
    'Data Config': DATA_YAML,
}

print("Copying deployment files:")
for name, src_path in files_to_copy.items():
    if src_path.exists():
        dst_path = deployment_dir / src_path.name
        shutil.copy2(src_path, dst_path)
        print(f"  ✓ {name}: {src_path.name}")
    else:
        print(f"  ⚠ {name}: Not found (may be generated during export)")

# Copy TFLite models if they exist
tflite_models = list(PROJECT_ROOT.glob('*.tflite'))
for tflite_model in tflite_models:
    dst_path = deployment_dir / tflite_model.name
    shutil.copy2(tflite_model, dst_path)
    print(f"  ✓ TFLite Model: {tflite_model.name}")

print(f"\n✓ Deployment package created at: {deployment_dir}")
print("=" * 60)

# Create README for deployment package
readme_content = f"""
# YOLOv11n ESP32 Deployment Package

## Package Contents
- `best.pt`: Best PyTorch model weights
- `best.onnx`: ONNX model (640x640 input)
- `yolo11n_int8_esp32.tflite`: INT8 quantized TFLite model (if available)
- `calibration_data_esp32.npy`: Calibration dataset for quantization
- `hyp_esp32_yolo11n.yaml`: Training hyperparameters
- `validation_metrics_esp32.json`: Model performance metrics
- `data.yaml`: Dataset configuration

## Model Specifications
- Architecture: YOLOv11n (Nano)
- Parameters: 2.6M
- Input Size: 640x640x3
- Task: Single-class detection (grape leaf)
- Quantization: INT8 (Post-Training Quantization)

## ESP32 Deployment
1. Follow ESP32_DEPLOYMENT_GUIDE.txt for detailed instructions
2. Use esp-dl tools to convert TFLite model to ESP-DL format
3. Flash to ESP32-S3 WROOM-1 or ESP32-P4 with NPU support
4. Expected inference time: 50-100ms on ESP32-P4

## Model Performance (Pre-Quantization)
- Validation mAP@0.5:0.95: See validation_metrics_esp32.json
- Training device: {'CUDA' if torch.cuda.is_available() else 'CPU'}
- Training date: {datetime.now().strftime('%Y-%m-%d')}

## References
- Espressif Detection: https://github.com/espressif/esp-detection
- Ultralytics YOLOv11: https://docs.ultralytics.com/models/yolo11/
- ESP-DL: https://github.com/espressif/esp-dl

## Support
For issues with ESP32 deployment, refer to:
- Espressif ESP32 Forum: https://esp32.com/
- ESP-DL GitHub Issues: https://github.com/espressif/esp-dl/issues
"""

readme_path = deployment_dir / 'README.md'
with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f"✓ README created at: {readme_path}")

In [ ]:
# Model Comparison Summary
print("YOLOv11n vs YOLOv5n Comparison for ESP32")
print("=" * 60)

comparison_table = """
╔══════════════════════╦═══════════════╦═══════════════╗
║ Metric               ║ YOLOv5n       ║ YOLOv11n      ║
╠══════════════════════╬═══════════════╬═══════════════╣
║ Parameters           ║ 1.9M          ║ 2.6M          ║
║ Model Size (FP32)    ║ ~7.5 MB       ║ ~10 MB        ║
║ Model Size (INT8)    ║ ~2 MB         ║ ~2.6 MB       ║
║ Input Resolution     ║ 640x640       ║ 640x640       ║
║ Architecture         ║ CSPDarknet    ║ C3k2 backbone ║
║ mAP Improvement      ║ Baseline      ║ +2-5%         ║
║ Inference (CPU)      ║ ~100ms        ║ ~120ms        ║
║ ESP32-S3 @ 640x640   ║ ~300-500ms    ║ ~400-600ms    ║
║ ESP32-S3 @ 416x416   ║ ~150-250ms    ║ ~200-300ms    ║
║ Training Time        ║ Faster        ║ Moderate      ║
║ Deployment           ║ Mature        ║ Newer         ║
╚══════════════════════╩═══════════════╩═══════════════╝

Key Advantages of YOLOv11n for ESP32-S3:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✓ Improved mAP@0.5:0.95 (better detection accuracy)
✓ Better small object detection (important for leaf features)
✓ More efficient C3k2 blocks (fewer FLOPs per parameter)
✓ Enhanced anchor-free detection (reduces post-processing)
✓ Better quantization resilience (INT8 accuracy retention)

Considerations for ESP32-S3 (NO NPU):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚠ Slightly larger model size (~30% more parameters)
⚠ Slower inference than YOLOv5n on ESP32-S3 (no NPU)
⚠ Newer architecture (fewer ESP-DL deployment examples)
⚠ May require custom ESP-DL layer implementations
⚠ Training time ~15% longer than YOLOv5n
⚠ Recommended to use 416x416 instead of 640x640 for real-time

Recommendation for ESP32-S3 WROOM-1:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ Use YOLOv11n if accuracy is top priority (better mAP)
→ Use YOLOv5n if speed/deployment simplicity is priority
→ CRITICAL: Reduce resolution to 416x416 for real-time performance
→ Expected real-world: 2-5 FPS @ 416x416, 1-2 FPS @ 640x640
→ Both models viable for vineyard monitoring (not real-time critical)
"""

print(comparison_table)

# Save comparison
comparison_path = PROJECT_ROOT / 'model_comparison_esp32.txt'
with open(comparison_path, 'w') as f:
    f.write(comparison_table)

print("=" * 60)
print(f"✓ Comparison saved to: {comparison_path}")

---

## Appendix: Troubleshooting & Advanced Optimizations

### Common Issues and Solutions

**1. Out of Memory (OOM) during training:**
- Reduce batch size (try 8 or 4)
- Reduce image size (512x512 or 416x416)
- Enable gradient checkpointing
- Use mixed precision training (but disable for PTQ)

**2. Low mAP scores:**
- Increase training epochs (200-300)
- Adjust learning rate schedule
- Enable more aggressive augmentation
- Check dataset quality and annotations

**3. TFLite conversion failures:**
- Use ONNX as intermediate format
- Ensure ONNX opset compatibility (opset 11-13)
- Verify TensorFlow version (2.15.0 recommended)
- Use onnx-tf converter with specific TF version

**4. ESP32 deployment issues:**
- Verify model size < 4MB (compressed)
- Check PSRAM allocation for input buffers
- Ensure INT8 quantization completed
- Test with ESP-DL example models first

**5. Poor quantization accuracy:**
- Increase calibration dataset size (200-500 samples)
- Use diverse calibration images (lighting, angles)
- Verify calibration data preprocessing matches training
- Consider Quantization-Aware Training (QAT) instead of PTQ

### Advanced Optimizations

**1. Reduce Model Size:**
```python
# Model pruning (requires additional libraries)
# torch.nn.utils.prune for structured pruning
# Recommended: 20-30% pruning for minimal accuracy loss
```

**2. Input Resolution Optimization:**
```python
# Test different resolutions for speed/accuracy trade-off
# 640x640: Best accuracy
# 512x512: Balanced
# 416x416: Fastest (recommended for ESP32-S3)
# 320x320: Real-time capable on ESP32-S3
```

**3. Knowledge Distillation:**
```python
# Train larger teacher model (YOLOv11m)
# Distill knowledge to YOLOv11n student
# Can improve student mAP by 1-3%
```

**4. Custom Layer Optimization:**
- Replace computationally expensive layers
- Implement custom ESP-DL kernels for specific operations
- Fuse BatchNorm into Conv layers for inference

**5. Dynamic Quantization:**
- Quantize only heavy layers (Conv, Dense)
- Keep lightweight layers in FP16
- Balance accuracy and speed

---

**End of YOLOv11n ESP32 Optimization Pipeline**

In [ ]:
# Final Summary and Next Steps
print("\n" + "=" * 60)
print("YOLOv11n ESP32 Training Pipeline - SUMMARY")
print("=" * 60)

summary = f"""
✓ COMPLETED STEPS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. Environment Setup
   └─ Ultralytics, ONNX, TensorFlow installed

2. Dataset Configuration
   └─ Single-class grape leaf detection
   └─ Training/Validation/Test splits verified

3. Model Training
   └─ YOLOv11n trained with ESP32-optimized hyperparameters
   └─ 150 epochs with early stopping (patience=30)
   └─ Conservative augmentation for edge stability

4. Validation
   └─ Comprehensive mAP evaluation on test set
   └─ Performance metrics saved to JSON

5. Model Export
   └─ ONNX format (opset 12)
   └─ TFLite INT8 quantization (if supported)
   └─ Calibration dataset generated (100 samples)

6. ESP32 Deployment Package
   └─ All necessary files collected
   └─ Deployment guide created
   └─ Ready for ESP-DL conversion

📊 EXPECTED PERFORMANCE (ESP32-S3 WROOM-1):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Model: YOLOv11n (2.6M parameters)
• Input: 640x640x3 RGB (trained) / 416x416 recommended for deployment
• Quantization: INT8 (Post-Training)
• Target Device: ESP32-S3 WROOM-1 (Dual-Core Xtensa LX7 @ 240MHz)
• Expected Inference: 
  - @ 640x640: ~400-600ms (1-2 FPS)
  - @ 416x416: ~200-300ms (3-5 FPS) ← RECOMMENDED
• Expected mAP@0.5:0.95: Competitive with YOLOv5n
• ⚠ NO NPU: ESP32-S3 uses vector instructions only

📁 OUTPUT FILES:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Model Weights: runs/detect/yolo11n_leaf_esp32/weights/best.pt
• ONNX Model: best.onnx
• TFLite Model: yolo11n_int8_esp32.tflite (if generated)
• Calibration: calibration_data_esp32.npy
• Metrics: validation_metrics_esp32.json
• Deployment Package: esp32_deployment_package/

🚀 NEXT STEPS FOR ESP32-S3 WROOM-1 DEPLOYMENT:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. Install ESP-IDF and ESP-DL:
   $ git clone --recursive https://github.com/espressif/esp-idf.git
   $ git clone https://github.com/espressif/esp-dl.git

2. Convert TFLite to ESP-DL format:
   $ cd esp-dl/tools
   $ python tflite2esp.py \\
       --model_path ../../yolo11n_int8_esp32.tflite \\
       --output_path yolo11n_esp32_s3.espdl \\
       --compress

3. Flash to ESP32-S3 WROOM-1:
   $ idf.py -p /dev/ttyUSB0 flash monitor

4. ESP32-S3 Performance Optimization:
   - Enable PSRAM (8MB) for model storage
   - Use vector instructions (Xtensa LX7 SIMD)
   - Optimize memory: SRAM for activations, PSRAM for weights
   - CRITICAL: Consider reducing to 416x416 input for real-time
   - Tune NMS parameters (IoU threshold, confidence)

5. Optional: Retrain for 416x416 resolution:
   - Change 'imgsz': 416 in hyperparameters (cell 6)
   - Re-run training cells
   - Expected: ~2x faster inference on ESP32-S3

6. Integration with ESP32-CAM:
   - Use ESP32-S3 camera module (OV2640/OV5640)
   - Capture at 800x600, crop to 416x416 center
   - Preprocess: normalize, convert to INT8
   - Run inference on ESP-DL model
   - Post-process: simplified NMS, threshold 0.25

📚 REFERENCES:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Espressif Detection: https://github.com/espressif/esp-detection
• ESP-DL Library: https://github.com/espressif/esp-dl
• ESP32-S3 Datasheet: https://www.espressif.com/sites/default/files/documentation/esp32-s3_datasheet_en.pdf
• Ultralytics Docs: https://docs.ultralytics.com/

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✓ Training and optimization pipeline complete!
✓ Ready for ESP32-S3 WROOM-1 deployment
⚠ Remember: ESP32-S3 has NO NPU - expect slower inference than ESP32-P4
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

print(summary)

# Save summary
summary_path = PROJECT_ROOT / 'TRAINING_SUMMARY.txt'
with open(summary_path, 'w') as f:
    f.write(summary)

print(f"✓ Training summary saved to: {summary_path}")
print("=" * 60)

## 7. Model Comparison & Summary
### YOLOv11n vs YOLOv5n for ESP32 Deployment

In [ ]:
# ESP-DL Model Conversion Guide for ESP32-S3 WROOM-1
# Reference: https://github.com/espressif/esp-dl

print("ESP32-S3 WROOM-1 Deployment Guide")
print("=" * 60)

deployment_guide = """
ESP32-S3 WROOM-1 Deployment Steps:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. PREREQUISITES:
   ├── ESP-IDF v5.1 or later
   ├── ESP-DL library (git clone https://github.com/espressif/esp-dl)
   ├── Calibrated INT8 TFLite model
   └── ESP32-S3 WROOM-1 Dev Kit with 8MB PSRAM

2. MODEL CONVERSION (TFLite → ESP-DL):
   ├── Use ESP-DL's model converter tool
   ├── Command: python esp-dl/tools/tflite2esp.py \\
   │            --model_path yolo11n_int8_esp32.tflite \\
   │            --output_path yolo11n_esp32_s3.espdl \\
   │            --compress
   └── This creates ESP-DL optimized binary for ESP32-S3

3. ESP32-S3 WROOM-1 DEPLOYMENT:
   ├── Flash model to SPIFFS/LittleFS partition (4MB recommended)
   ├── Enable PSRAM for model storage: menuconfig → Component config → ESP32-S3-Specific
   ├── Sample code: esp-dl/examples/human_face_detection
   ├── Adapt for single-class YOLO detection
   └── Use esp-dl/components/modules for detection layers

4. RESOLUTION OPTIMIZATION (CRITICAL for ESP32-S3):
   ├── 640x640: ~400-600ms inference (training resolution)
   ├── 416x416: ~200-300ms inference (RECOMMENDED for real-time)
   ├── 320x320: ~150-200ms inference (fastest, lower accuracy)
   └── Trade-off: Smaller resolution = faster but less accurate

5. ESP32-S3 OPTIMIZATION TECHNIQUES:
   ├── Enable vector instructions in ESP-IDF (Xtensa LX7 SIMD)
   ├── Input preprocessing: RGB565 → INT8 (on-device conversion)
   ├── Use PSRAM for model weights, SRAM for activations
   ├── Enable DMA for camera → PSRAM transfers
   ├── Quantization-aware inference (INT8 throughout)
   └── Batch size = 1 (single image inference only)

6. PERFORMANCE MONITORING:
   ├── Use ESP-IDF performance monitoring tools
   ├── Monitor: inference time, memory usage, power consumption
   └── Realistic ESP32-S3 targets:
       • Inference @ 416x416: ~200-300ms
       • Memory: < 512KB SRAM, ~3-4MB PSRAM
       • Power: ~400-600mW during inference
       • FPS: 2-5 FPS (sufficient for vineyard monitoring)

7. INTEGRATION WITH ESP32-CAM:
   ├── Use ESP32-S3 camera module (OV2640/OV5640)
   ├── Capture at lower resolution (800x600), crop to 416x416
   ├── Preprocess on ESP32-S3: resize, normalize to INT8
   ├── Run inference on preprocessed frame
   ├── Post-process: Simplified NMS, confidence threshold 0.25
   └── Stream results via WiFi/MQTT for monitoring

8. MEMORY OPTIMIZATION:
   ├── Use model compression (quantization + pruning)
   ├── Load model weights from PSRAM (slower but feasible)
   ├── Reuse activation buffers between layers


   └── Consider layer-by-layer execution to reduce peak memoryprint("\n⚠ IMPORTANT: Consider retraining at 416x416 for better ESP32-S3 performance")

# Save ESP32-S3 specific recommendations
"""print(f"\n✓ ESP32-S3 specific notes saved to: {esp32_s3_notes_path}")

esp32_s3_notes = """


ESP32-S3 WROOM-1 SPECIFIC NOTES:
print(deployment_guide)    f.write(esp32_s3_notes)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("=" * 60)with open(esp32_s3_notes_path, 'w') as f:


esp32_s3_notes_path = PROJECT_ROOT / 'ESP32_S3_NOTES.txt'

⚠ IMPORTANT: ESP32-S3 does NOT have dedicated NPU/AI accelerator
# Save deployment guide

   - Relies on Xtensa LX7 vector instructions for acceleration
deployment_path = PROJECT_ROOT / 'ESP32_DEPLOYMENT_GUIDE.txt'"""

   - Inference will be slower than ESP32-P4 (no NPU)
with open(deployment_path, 'w') as f:   - Trade-off: ~2-3% lower mAP

   - Expect 200-500ms @ 640x640, 150-300ms @ 416x416
    f.write(deployment_guide)   - Slightly faster inference (fewer parameters: 1.9M vs 2.6M)


   - More mature ESP-DL examples available

✅ RECOMMENDED CONFIGURATION for Real-Time Performance:
print(f"✓ Deployment guide saved to: {deployment_path}")💡 ALTERNATIVE: Use YOLOv5n for better ESP32-S3 support

   - Input resolution: 416x416 (instead of 640x640)

   - Quantization: INT8 (mandatory for reasonable speed)   4. Expected speedup: ~2x faster on ESP32-S3

   - Model: YOLOv11n (already smallest YOLO variant)   3. Re-export to ONNX/TFLite with imgsz=416

   - Frame rate target: 2-5 FPS (sufficient for vineyard monitoring)   2. Retrain model from scratch or fine-tune

   1. Change 'imgsz': 416 in hyperparameters
🔧 TO RETRAIN FOR 416x416 (if needed):

In [ ]:
# Alternative Method: Direct TFLite INT8 Export from YOLOv11
# This method uses Ultralytics' built-in conversion (if available in newer versions)

print("Alternative: Direct TFLite INT8 Export...")
print("=" * 60)

try:
    # Attempt direct INT8 export (requires Ultralytics 8.1.0+)
    tflite_int8_path = model_best.export(
        format='tflite',
        imgsz=640,
        int8=True,  # Enable INT8 quantization
        data=str(DATA_YAML),  # Provide dataset for calibration
        nms=False,  # Disable NMS for faster inference
        simplify=True,
    )
    
    print(f"✓ Direct INT8 TFLite export successful!")
    print(f"✓ Model saved to: {tflite_int8_path}")
    
    # Get model size
    model_size = Path(tflite_int8_path).stat().st_size
    print(f"✓ Model size: {model_size / 1024:.2f} KB ({model_size / (1024*1024):.2f} MB)")
    print("=" * 60)
    
except Exception as e:
    print(f"⚠ Direct INT8 export not available or failed: {e}")
    print("Use the ONNX model with manual TFLite conversion")
    print("Refer to Espressif detection repo for conversion scripts")
    print("=" * 60)

In [ ]:
# Step 4: Post-Training Quantization to INT8 TFLite
# Following Espressif detection repository guidelines

print("Performing Post-Training INT8 Quantization...")
print("=" * 60)

import tensorflow as tf

def representative_dataset_gen():
    """
    Generator function for representative dataset
    Required for INT8 quantization
    """
    # Load calibration data
    calib_data = np.load(calib_path)
    
    for i in range(min(100, len(calib_data))):
        # Yield single sample as float32
        yield [calib_data[i:i+1].astype(np.float32)]

if savedmodel_path and savedmodel_path.exists():
    try:
        # Convert SavedModel to TFLite with INT8 quantization
        converter = tf.lite.TFLiteConverter.from_saved_model(str(savedmodel_path))
        
        # Enable INT8 quantization
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset_gen
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.uint8  # or tf.int8
        converter.inference_output_type = tf.uint8  # or tf.int8
        
        # Convert model
        tflite_quant_model = converter.convert()
        
        # Save quantized model
        tflite_quant_path = PROJECT_ROOT / 'yolo11n_int8_esp32.tflite'
        with open(tflite_quant_path, 'wb') as f:
            f.write(tflite_quant_model)
        
        print(f"✓ INT8 TFLite model saved to: {tflite_quant_path}")
        print(f"✓ Model size: {len(tflite_quant_model) / 1024:.2f} KB")
        print("=" * 60)
        
    except Exception as e:
        print(f"⚠ INT8 quantization failed: {e}")
        print("Attempting FP16 quantization as fallback...")
        
        # Fallback to FP16 quantization
        converter = tf.lite.TFLiteConverter.from_saved_model(str(savedmodel_path))
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
        
        tflite_fp16_model = converter.convert()
        tflite_fp16_path = PROJECT_ROOT / 'yolo11n_fp16_esp32.tflite'
        with open(tflite_fp16_path, 'wb') as f:
            f.write(tflite_fp16_model)
        
        print(f"✓ FP16 TFLite model saved to: {tflite_fp16_path}")
        print(f"✓ Model size: {len(tflite_fp16_model) / 1024:.2f} KB")
        print("=" * 60)
else:
    print("⚠ SavedModel not available - skipping TFLite INT8 quantization")
    print("Please use the ONNX model for manual conversion")
    print("=" * 60)

In [ ]:
# Step 2: Generate Calibration Dataset for INT8 Quantization
# Using representative images from training set

print("Generating Calibration Dataset for INT8 Quantization...")
print("=" * 60)

def create_calibration_dataset(image_dir, num_samples=100, target_size=(640, 640)):
    """
    Create calibration dataset for INT8 quantization
    Following Espressif detection guidelines
    """
    image_files = list(Path(image_dir).glob('*.jpg'))[:num_samples]
    calibration_data = []
    
    for img_path in tqdm(image_files, desc="Preparing calibration data"):
        # Load and preprocess image
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, target_size)
        
        # Normalize to [0, 1] and convert to float32
        img = img.astype(np.float32) / 255.0
        calibration_data.append(img)
    
    # Stack into batch and transpose to NCHW format
    calibration_batch = np.stack(calibration_data, axis=0)
    calibration_batch = np.transpose(calibration_batch, (0, 3, 1, 2))  # NHWC -> NCHW
    
    return calibration_batch

# Create calibration dataset
calib_data = create_calibration_dataset(
    DATASET_PATH / 'train' / 'images',
    num_samples=100
)

print(f"\nCalibration Dataset:")
print(f"├── Shape: {calib_data.shape}")
print(f"├── Data Type: {calib_data.dtype}")
print(f"├── Min Value: {calib_data.min():.4f}")
print(f"├── Max Value: {calib_data.max():.4f}")
print(f"└── Mean Value: {calib_data.mean():.4f}")

# Save calibration data for later use
calib_path = PROJECT_ROOT / 'calibration_data_esp32.npy'
np.save(calib_path, calib_data)
print(f"\n✓ Calibration data saved to: {calib_path}")
print("=" * 60)

## 5. Model Export & Post-Training Quantization (PTQ)
### ESP32-S3 & ESP32-P4 NPU Optimization Pipeline

**Export Formats:**
1. **ONNX** (Intermediate format for optimization)
2. **TensorFlow Lite INT8** (Post-Training Quantization for ESP32)
3. **ESP-DL Format** (Espressif's native format for NPU acceleration)

In [ ]:
# Inference Speed Benchmark (Pre-Quantization Baseline)
import time

print("Inference Speed Benchmark (PyTorch FP32):")
print("=" * 60)

# Load test images
test_images = list((DATASET_PATH / 'test' / 'images').glob('*.jpg'))[:20]

inference_times = []
for img_path in tqdm(test_images, desc="Benchmarking"):
    start_time = time.time()
    results = model_best(str(img_path), imgsz=640, conf=0.25, verbose=False)
    inference_times.append((time.time() - start_time) * 1000)  # Convert to ms

avg_inference_time = np.mean(inference_times)
std_inference_time = np.std(inference_times)
fps = 1000 / avg_inference_time

print(f"\nPyTorch FP32 Performance (Pre-Quantization):")
print(f"├── Average Inference Time: {avg_inference_time:.2f} ± {std_inference_time:.2f} ms")
print(f"├── FPS: {fps:.2f}")
print(f"├── Min Time: {np.min(inference_times):.2f} ms")
print(f"└── Max Time: {np.max(inference_times):.2f} ms")
print("\n⚠ Note: ESP32 performance will be different after quantization")
print("=" * 60)

## 4. Model Validation & Performance Evaluation
### Comprehensive mAP Analysis for ESP32 Deployment

In [ ]:
# Training Execution with ESP32-Optimized Configuration
# Following Espressif detection guidelines for edge deployment

print("Starting YOLOv11n Training for ESP32 Deployment...")
print("=" * 60)

# Train the model
results = model.train(
    data=str(DATA_YAML),
    epochs=esp32_hyperparameters['epochs'],
    imgsz=esp32_hyperparameters['imgsz'],
    batch=esp32_hyperparameters['batch'],
    patience=esp32_hyperparameters['patience'],
    
    # Optimizer settings
    optimizer=esp32_hyperparameters['optimizer'],
    lr0=esp32_hyperparameters['lr0'],
    lrf=esp32_hyperparameters['lrf'],
    momentum=esp32_hyperparameters['momentum'],
    weight_decay=esp32_hyperparameters['weight_decay'],
    
    # Warmup settings
    warmup_epochs=esp32_hyperparameters['warmup_epochs'],
    warmup_momentum=esp32_hyperparameters['warmup_momentum'],
    warmup_bias_lr=esp32_hyperparameters['warmup_bias_lr'],
    
    # Augmentation settings
    hsv_h=esp32_hyperparameters['hsv_h'],
    hsv_s=esp32_hyperparameters['hsv_s'],
    hsv_v=esp32_hyperparameters['hsv_v'],
    degrees=esp32_hyperparameters['degrees'],
    translate=esp32_hyperparameters['translate'],
    scale=esp32_hyperparameters['scale'],
    shear=esp32_hyperparameters['shear'],
    perspective=esp32_hyperparameters['perspective'],
    flipud=esp32_hyperparameters['flipud'],
    fliplr=esp32_hyperparameters['fliplr'],
    mosaic=esp32_hyperparameters['mosaic'],
    mixup=esp32_hyperparameters['mixup'],
    copy_paste=esp32_hyperparameters['copy_paste'],
    
    # Loss configuration
    box=esp32_hyperparameters['box'],
    cls=esp32_hyperparameters['cls'],
    dfl=esp32_hyperparameters['dfl'],
    
    # Training settings
    device=esp32_hyperparameters['device'],
    workers=esp32_hyperparameters['workers'],
    project='runs/detect',
    name='yolo11n_leaf_esp32',
    exist_ok=esp32_hyperparameters['exist_ok'],
    pretrained=esp32_hyperparameters['pretrained'],
    verbose=esp32_hyperparameters['verbose'],
    plots=esp32_hyperparameters['plots'],
    val=esp32_hyperparameters['val'],
    save=esp32_hyperparameters['save'],
    
    # ESP32 optimization flags
    half=esp32_hyperparameters['half'],
    amp=esp32_hyperparameters['amp'],
    multi_scale=esp32_hyperparameters['multi_scale'],
)

print("\n" + "=" * 60)
print("✓ Training completed successfully!")
print(f"✓ Best model saved to: {model.trainer.best}")
print("=" * 60)

## 3. Model Initialization & Training
### YOLOv11n Training with ESP32 Deployment Constraints

In [ ]:
# Visualize Sample Training Images with Annotations
def visualize_yolo_annotations(image_dir, label_dir, num_samples=6):
    """Visualize YOLO format annotations on images"""
    image_files = list(Path(image_dir).glob('*.jpg'))[:num_samples]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(image_files):
        # Load image
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # Load corresponding label
        label_path = Path(label_dir) / f"{img_path.stem}.txt"
        
        if label_path.exists():
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls, x_center, y_center, width, height = map(float, parts[:5])
                        
                        # Convert YOLO format to pixel coordinates
                        x1 = int((x_center - width/2) * w)
                        y1 = int((y_center - height/2) * h)
                        x2 = int((x_center + width/2) * w)
                        y2 = int((y_center + height/2) * h)
                        
                        # Draw bounding box
                        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        cv2.putText(img, 'leaf', (x1, y1-10), 
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        axes[idx].imshow(img)
        axes[idx].set_title(f"{img_path.name}", fontsize=10)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('dataset_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Dataset visualization saved as 'dataset_samples.png'")

# Visualize training samples
visualize_yolo_annotations(
    DATASET_PATH / 'train' / 'images',
    DATASET_PATH / 'train' / 'labels',
    num_samples=6
)

## 2. Dataset Preparation & Configuration
### Loading Single-Class Grape Leaf Detection Dataset

In [ ]:
# Install Required Packages
# Ultralytics YOLOv11, ONNX Runtime, TensorFlow Lite, and ESP-DL utilities

!pip install ultralytics>=8.3.0 --quiet
!pip install onnx>=1.15.0 onnxruntime>=1.16.0 --quiet
!pip install tensorflow==2.15.0 --quiet  # For TFLite conversion
!pip install onnx-tf --quiet  # ONNX to TensorFlow conversion bridge
!pip install matplotlib seaborn opencv-python-headless --quiet
!pip install pillow albumentations --quiet  # Advanced augmentation
!pip install pyyaml tqdm --quiet

print("✓ All dependencies installed successfully")

## 1. Environment Configuration
### Installing Dependencies for YOLOv11 and ESP32 Deployment